In [2]:
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
from env import (MultiLeafThreadEnv, ThreadingConfig)
from viz import  draw_tree_edge_index, draw_local_tree_sequence
from utils import build_backbone_segments_from_reference, get_max_actions, MultiLeafState, load_tskit_threading_inputs
from models import Policy

In [3]:
# TREES_PATH = "path/to/file.trees"
# TIME_GRID_SIZE = 20

# GENO, REFERENCE_FULL_TREES, LEAF_NAMES, ALL_LEAF_IDS, TIME_GRID = load_tskit_threading_inputs(
#     TREES_PATH,
#     time_grid_size=TIME_GRID_SIZE,
# )

## First example
REFERENCE_FULL_TREES = [
    {
        "sites": (0, 1),
        # "tree": ("n", 2.0, ("n", 1.0, ("n", 1.0, 0, 1), 2), 3),
        "tree": ("n", 2.0, ("n", 0.25, 0, 1), ("n", 0.25, 2, 3)),
    },
    {
        "sites": (1, 8),
        "tree": ("n", 2.0, ("n", 0.25, 0, 2), ("n", 0.25, 1, 3)),
    },
]

GENO = torch.tensor(
    [
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
    ],
    dtype=torch.long,
)
LEAF_NAMES = ["A", "B", "C", "D"]
ALL_LEAF_IDS = [0, 1, 2, 3]
TIME_GRID = (0.25, 0.5, 1.0, 2.0, 4.0)

In [ ]:
## View this tree. 
full_tree_backbone = build_backbone_segments_from_reference(REFERENCE_FULL_TREES, focal_leaf=-1)
draw_local_tree_sequence(full_tree_backbone, leaf_names=LEAF_NAMES, time_grid=TIME_GRID, use_time_as_y=True)

In [ ]:
env_cfg = ThreadingConfig.from_raw(GENO, TIME_GRID, 0.4, 0.35, 0.15)
env = MultiLeafThreadEnv(env_cfg, ALL_LEAF_IDS, REFERENCE_FULL_TREES)

In [ ]:
st = env.reset() ## Resetting the state. Meaning one sequence is removed from here. 

In [ ]:
env_cfg

In [ ]:
env._inner_env._site_tree_for_encoding(st.inner_state)


In [ ]:
## encoding is done here. We need to make changes here. 
## Here, there's need to properly initialize the node features and valid action masks. 
## No need for genotype-window at the encoding step. It'll be used during the reward for reparamaterization. 
# encoded_state = env.encode(st)

In [ ]:
policy_model = Policy(LEAF_NAMES, device='cpu')

In [ ]:
site_tree = env._inner_env._site_tree_for_encoding(st.inner_state)
action_choices = env._inner_env.site_choices[st.inner_state.site_index]
action_logits, action_probs, edge_features, node_embeddings, leaf_feature = policy_model(
    site_tree,
    st.current_focal_leaf,
    action_choices,
    time_grid=env_cfg.time_grid,
)

In [ ]:
act = int(np.argmax(action_probs.detach().numpy()))

In [ ]:
st, reward, done = env.step(st, act)

In [ ]:
st 

In [ ]:
fina

In [ ]:
## One episode REINFORCE loop with additive local log rewards
## Shows action probabilities, per-action local rewards, and the local tree sequence.

if "optimizer" not in globals():
    optimizer = torch.optim.Adam(policy_model.parameters(), lr=1e-3)

policy_model.train()
st = env.reset()
saved_log_probs = []
episode_actions = []
step_log_rewards = []
done = False
step_idx = 0


def _choice_key(choice):
    return choice.branch_child, choice.branch_signature


while not done:
    assert env._inner_env is not None and st.inner_state is not None
    inner_env = env._inner_env
    inner_state = st.inner_state
    site_idx = inner_state.site_index

    valid_acts = env.valid_actions(st)
    if not valid_acts:
        raise RuntimeError(f"No valid actions at episode step {step_idx}")

    # Keep action_choices aligned with valid_acts: row j in action_probs maps to valid_acts[j].
    action_choices = [inner_env.site_choices[site_idx][action_idx] for action_idx in valid_acts]
    site_tree = inner_env._site_tree_for_encoding(inner_state)
    action_logits, action_probs, edge_features, node_embeddings, leaf_feature = policy_model(
        site_tree,
        st.current_focal_leaf,
        action_choices,
        time_grid=env_cfg.time_grid,
    )

    focal_name = LEAF_NAMES[st.current_focal_leaf]
    print(f"\n=== episode step {step_idx} | leaf {focal_name} | site {site_idx} ===")
    print("Action probabilities:")
    probs_cpu = action_probs.detach().cpu()
    for row_idx, (action_idx, prob) in enumerate(zip(valid_acts, probs_cpu)):
        desc = env.describe_action(st, action_idx, leaf_names=LEAF_NAMES)
        print(f"  row {row_idx:02d} -> env action {action_idx:02d} | p={prob.item():.4f} | {desc}")

    dist = torch.distributions.Categorical(probs=action_probs)
    selected_row = dist.sample()
    selected_row_idx = int(selected_row.item())
    selected_action = valid_acts[selected_row_idx]
    selected_choice = action_choices[selected_row_idx]
    selected_desc = env.describe_action(st, selected_action, leaf_names=LEAF_NAMES)
    saved_log_probs.append(dist.log_prob(selected_row))
    print(f"Selected: row {selected_row_idx} -> env action {selected_action} | {selected_desc}")

    # Build a preview state for visualization before env.step potentially advances to the next leaf.
    recomb_count = inner_state.recomb_count
    if inner_state.choices and _choice_key(selected_choice) != _choice_key(inner_state.choices[-1]):
        recomb_count += 1
    preview_inner_state = type(inner_state)(
        site_index=site_idx + 1,
        choices=inner_state.choices + (selected_choice,),
        recomb_count=recomb_count,
    )

    st, local_log_reward, done = env.step(st, selected_action)
    step_log_rewards.append(float(local_log_reward))
    print(
        f"Local log reward: {local_log_reward:.4f} | "
        f"positive reward factor: {float(np.exp(local_log_reward)):.4e}"
    )

    episode_actions.append(
        {
            "step": step_idx,
            "leaf": focal_name,
            "site": site_idx,
            "row": selected_row_idx,
            "env_action": int(selected_action),
            "prob": float(probs_cpu[selected_row_idx].item()),
            "local_log_reward": float(local_log_reward),
            "description": selected_desc,
        }
    )

    local_trees = inner_env.snapshot_state(preview_inner_state)
    if local_trees:
        print("Local tree sequence after selected action:")
        draw_local_tree_sequence(
            local_trees,
            leaf_names=LEAF_NAMES,
            time_grid=TIME_GRID,
            use_time_as_y=True,
        )
        plt.show()
    else:
        print("No local tree sequence yet.")

    step_idx += 1

log_probs_t = torch.stack(saved_log_probs)
step_log_rewards_t = torch.tensor(step_log_rewards, dtype=log_probs_t.dtype, device=log_probs_t.device)
reward_to_go = torch.flip(
    torch.cumsum(torch.flip(step_log_rewards_t, dims=[0]), dim=0),
    dims=[0],
)
total_log_reward = step_log_rewards_t.sum()
loss = -(reward_to_go.detach() * log_probs_t).sum()

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("\n=== episode summary ===")
print(f"steps: {step_idx}")
print(f"total additive log reward: {total_log_reward.item():.4f}")
print(f"loss: {loss.item():.4f}")
print(f"leaves threaded: {st.leaves_threaded}")
print("selected actions:")
for action in episode_actions:
    print(
        f"  step {action['step']:02d} | leaf {action['leaf']} | site {action['site']} | "
        f"row {action['row']:02d} -> env action {action['env_action']:02d} | "
        f"p={action['prob']:.4f} | local_log_reward={action['local_log_reward']:.4f} | "
        f"{action['description']}"
    )

## Misc..

In [ ]:
# --- Valid-action edge features (focal leaf x attachment target x time) ---


def extract_valid_action_choices(
    env, st, valid_action_indices: list[int]
):
    """Map flat action indices to :class:`ThreadChoice` and stacked tensors."""
    assert env._inner_env is not None and st.inner_state is not None
    site_choices = env._inner_env.site_choices[st.inner_state.site_index]
    choices = [site_choices[i] for i in valid_action_indices]
    return choices


def valid_action_choices_to_tensors(choices, device, dtype=torch.float32):
    branch_child = torch.tensor(
        [c.branch_child for c in choices], dtype=torch.long, device=device
    )
    time_idx = torch.tensor(
        [c.time_idx for c in choices], dtype=torch.long, device=device
    )
    time_value = torch.tensor(
        [float(c.time_value) for c in choices], dtype=dtype, device=device
    )
    is_root = torch.tensor(
        [1.0 if c.is_root_branch else 0.0 for c in choices],
        dtype=dtype,
        device=device,
    )
    return branch_child, time_idx, time_value, is_root


def normalize_time_values(
    time_value: torch.Tensor, time_grid: tuple
) -> torch.Tensor:
    """[0,1] normalized time w.r.t. grid min/max (matches plan: scalar time)."""
    tg = torch.tensor(time_grid, dtype=time_value.dtype, device=time_value.device)
    t_min, t_max = tg[0], tg[-1]
    return (time_value - t_min) / (t_max - t_min + 1e-8)

In [ ]:
import torch.nn as nn


def build_valid_action_edge_features(
    node_embeddings: torch.Tensor,
    leaf_feature: torch.Tensor,
    policy_model: Policy,
    branch_child: torch.Tensor,
    time_value: torch.Tensor,
    is_root: torch.Tensor,
    time_grid: tuple,
) -> torch.Tensor:
    """
    h_e = f({h_focal, h_v}) with permutation-invariant MLP input:
    concat( h_f, h_v, |h_f-h_v|, h_f * h_v, t_norm, is_root ).
    h_f = focal_proj(one_hot_leaf), h_v = GNN node embedding of attachment branch child.
    """
    device, dtype = node_embeddings.device, node_embeddings.dtype
    leaf_feature = leaf_feature.to(device=device, dtype=dtype)
    h_f = policy_model.focal_proj(leaf_feature.unsqueeze(0)).squeeze(0)  # (D,)
    h_v = node_embeddings[branch_child]  # (A, D)
    h_f_b = h_f.unsqueeze(0).expand_as(h_v)
    t_norm = normalize_time_values(time_value, time_grid).unsqueeze(-1).to(dtype=dtype)
    h_e = torch.cat(
        [h_f_b, h_v, (h_f_b - h_v).abs(), h_f_b * h_v, t_norm, is_root.unsqueeze(-1)],
        dim=-1,
    )
    return h_e


def assert_valid_action_edge_alignment(
    env, st, valid_action_indices, choices, branch_child, h_e, site_tree_num_nodes: int
):
    assert h_e.shape[0] == len(
        valid_action_indices
    ), f"edge rows {h_e.shape[0]} != len(valid) {len(valid_action_indices)}"
    inner = env._inner_env
    sc = inner.site_choices[st.inner_state.site_index]
    for i, aidx in enumerate(valid_action_indices):
        assert sc[aidx] == choices[i]
    assert (branch_child >= 0).all() and (branch_child < site_tree_num_nodes).all()

In [ ]:
# --- One-shot pipeline (re-run to sample a fresh start state) ---
st = env.reset()
valid_acts = env.valid_actions(st)
assert len(valid_acts) > 0, "need non-terminal state with valid actions"

site_tree = env._inner_env._site_tree_for_encoding(st.inner_state)
choices = extract_valid_action_choices(env, st, valid_acts)
action_logits, action_probs, edge_features, node_embeddings, leaf_feature = policy_model(
    site_tree,
    st.current_focal_leaf,
    choices,
    time_grid=env_cfg.time_grid,
)

branch_child, time_idx, time_value, is_root = valid_action_choices_to_tensors(
    choices, device=node_embeddings.device, dtype=node_embeddings.dtype
)
assert_valid_action_edge_alignment(
    env,
    st,
    valid_acts,
    choices,
    branch_child,
    edge_features,
    site_tree_num_nodes=site_tree.num_nodes,
)
print("edge feature matrix:", edge_features.shape, "(rows align with valid_acts in order)")
print("probs sum:", action_probs.sum().item())

K = min(5, action_probs.shape[0])
print(f"\nTop {K} actions (policy action head):")
for rank, j in enumerate(torch.topk(action_probs, K).indices.tolist()):
    aidx = valid_acts[j]
    print(
        f"  {rank+1}. p={action_probs[j].item():.4f}  aidx={aidx}  {env.describe_action(st, aidx, leaf_names=LEAF_NAMES)}"
    )
    print(
        f"        branch_child={choices[j].branch_child}  time_idx={choices[j].time_idx}  t={choices[j].time_value}"
    )

In [ ]:
st = env.reset()
valid_acts = env.valid_actions(st)
print("number of valid actions for the given input ARG .. local tree is", len(valid_acts))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

## Possible Actions space for single Site

def plot_action(i):
    st = env.reset()
    k = env.describe_action(st, valid_acts[i], leaf_names=LEAF_NAMES)
    print(k)
    
    choice = env._inner_env.site_choices[st.inner_state.site_index][valid_acts[i]]
    print(choice)
    # print(choice.branch_signature, choice.time_value)
    st, reward, done = env.step(st, valid_acts[i])
    local_trees = env._inner_env.snapshot_state(st.inner_state)
    draw_local_tree_sequence(local_trees, leaf_names=LEAF_NAMES, time_grid=TIME_GRID, use_time_as_y=True)

widgets.interact(plot_action, i=widgets.IntSlider(min=0, max=len(valid_acts)-1, step=1, value=0, description='Action Index'))

In [ ]:
## Policy 
# policy = ARGForwardPolicy(initializing parameters)

reward = []
## I feel like there's no need for encoding step. 
## Just send the state directly to forward function of the policy and 
## it should output all the probs to attach to. 
action_probs, masked_logits = policy(st)
action_idx = max(action_probs) ## choose the action
next_st, action_reward, done = env.step(st, action)

if done:
    reward.append(action_reward)